<div style="background:linear-gradient(120deg,#EDE7FF 0%,#E3F0FF 55%,#E6FBF7 100%); color:#2A1B5C; padding:22px 26px; border-radius:14px; border:1px solid #DCD3FF; margin-bottom:8px;">

<h1 style="margin:0 0 6px 0; color:#3B1F7A">02 · Part D 2024 → GLP-1 subset</h1>

<p style="margin:0 0 10px 0; font-size:1.1em; color:#4C3A8C">Filtering a 28-million-row government file with DuckDB, then a first look at the US GLP-1 market</p>

| | |
|---|---|
| **Goal** | Turn the 4.4 GB CMS *Medicare Part D Prescribers by Provider and Drug, 2024* file into a small, clean GLP-1 table that later notebooks load in a second |
| **Input** | `data/01_raw/medicare_part_d_prescribers/…_2024.csv` (28,023,892 rows, never edited) |
| **Output** | `data/02_interim/part_d_2024_glp1.parquet` (420,364 rows, 11 MB) |
| **Skills** | DuckDB · SQL · Parquet · data validation · first EDA |

</div>

<div style="margin:6px 0 4px 0"><b>How to read this notebook:</b><br><span style="background:#F3EEFF; color:#3B1F7A; border-left:4px solid #8B5CF6; padding:3px 10px; border-radius:6px; margin-right:6px; display:inline-block; margin-bottom:4px">🟣 Section</span><span style="background:#FFF4E8; color:#7C2D12; border-left:4px solid #FB923C; padding:3px 10px; border-radius:6px; margin-right:6px; display:inline-block; margin-bottom:4px">💡 Concept</span><span style="background:#E9FBF8; color:#134E4A; border-left:4px solid #14B8A6; padding:3px 10px; border-radius:6px; margin-right:6px; display:inline-block; margin-bottom:4px">🛠️ What the next code does</span><span style="background:#EAF4FF; color:#1E3A8A; border-left:4px solid #3B82F6; padding:3px 10px; border-radius:6px; margin-right:6px; display:inline-block; margin-bottom:4px">📊 Reading the output</span><span style="background:#FFF0F3; color:#881337; border-left:4px solid #F43F5E; padding:3px 10px; border-radius:6px; margin-right:6px; display:inline-block; margin-bottom:4px">⚠️ Watch out</span><span style="background:#EAFBF1; color:#065F46; border-left:4px solid #10B981; padding:3px 10px; border-radius:6px; margin-right:6px; display:inline-block; margin-bottom:4px">🎯 Key takeaways</span></div>

<div style="border-left:6px solid #FB923C; background:#FFF4E8; color:#7C2D12; padding:14px 18px; border-radius:10px; margin:10px 0; box-shadow:0 1px 3px rgba(0,0,0,.06);">

<b style="font-size:1.02em; letter-spacing:.3px">💡 Concept · What is in this file?</b>

**One row = one prescriber × one drug × one year.**
Example: *"Dr X in Ohio wrote 120 Mounjaro claims for 25 Medicare patients in 2024, costing US$150k."*

| Column | Meaning |
|---|---|
| `Prscrbr_NPI` | National Provider Identifier: the doctor's unique 10-digit ID (our join key everywhere) |
| `Prscrbr_Type` | Specialty, e.g. Endocrinology, Family Practice |
| `Tot_Clms` | Number of prescriptions **including refills** |
| `Tot_Benes` | Number of distinct patients (beneficiaries) |
| `Tot_Drug_Cst` | Total paid (US$), **before** manufacturer rebates |
| `GE65_*` | The same numbers for patients aged 65+ |

</div>

<div style="border-left:6px solid #8B5CF6; background:#F3EEFF; color:#3B1F7A; padding:14px 18px; border-radius:10px; margin:10px 0; box-shadow:0 1px 3px rgba(0,0,0,.06);">

## 1 · Setup: why DuckDB, not pandas?

</div>

<div style="border-left:6px solid #FB923C; background:#FFF4E8; color:#7C2D12; padding:14px 18px; border-radius:10px; margin:10px 0; box-shadow:0 1px 3px rgba(0,0,0,.06);">

<b style="font-size:1.02em; letter-spacing:.3px">💡 Concept</b>

| | pandas `read_csv` | DuckDB |
|---|---|---|
| **Memory** | Loads *everything* into RAM. 4.4 GB of text becomes ~10–20 GB of Python objects, which crashes a laptop | **Streams** the file in chunks, so memory stays small |
| **Storage model** | Row by row | **Columnar + vectorised**: reads only the columns a query needs, ~2,000 values at a time |
| **Speed** | One CPU core | Uses **all cores in parallel** |
| **Language** | Python methods | **SQL**, the language of every data warehouse (Snowflake, BigQuery, Databricks) |
| **Setup** | — | No server: an *in-process* analytics database (like SQLite, but built for OLAP) |

**Rule of thumb:** use a query engine to **reduce** big data first, then use pandas on the small result.

</div>

<div style="border-left:6px solid #14B8A6; background:#E9FBF8; color:#134E4A; padding:14px 18px; border-radius:10px; margin:10px 0; box-shadow:0 1px 3px rgba(0,0,0,.06);">

<b style="font-size:1.02em; letter-spacing:.3px">🛠️ What the next code does</b>

1. Import libraries and our colour-coded table helper `show()`.
2. Find the project root, so the notebook works from any folder.
3. Open an in-memory DuckDB connection (nothing is saved unless we write it out).

</div>

In [1]:
# ── Imports ─────────────────────────────────────────────────────────────────
from pathlib import Path
import time

import duckdb
import pandas as pd

from glp1_intel.notebook_style import show          # light-blue output tables

# ── Paths ───────────────────────────────────────────────────────────────────
# Walk up from the current folder until we find the one that contains data/01_raw
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data" / "01_raw").exists())

RAW_CSV     = ROOT / "data/01_raw/medicare_part_d_prescribers/medicare_part_d_prescribers_by_provider_and_drug_2024.csv"
OUT_PARQUET = ROOT / "data/02_interim/part_d_2024_glp1.parquet"

# ── DuckDB connection ───────────────────────────────────────────────────────
con = duckdb.connect()                               # in-memory database
con.sql("SET enable_progress_bar = false")           # keep notebook output clean

print(f"CSV size: {RAW_CSV.stat().st_size / 1e9:.2f} GB")

CSV size: 4.43 GB


<div style="border-left:6px solid #8B5CF6; background:#F3EEFF; color:#3B1F7A; padding:14px 18px; border-radius:10px; margin:10px 0; box-shadow:0 1px 3px rgba(0,0,0,.06);">

## 2 · Look before you load: the schema

</div>

<div style="border-left:6px solid #FB923C; background:#FFF4E8; color:#7C2D12; padding:14px 18px; border-radius:10px; margin:10px 0; box-shadow:0 1px 3px rgba(0,0,0,.06);">

<b style="font-size:1.02em; letter-spacing:.3px">💡 Concept</b>

`DESCRIBE` makes DuckDB **sniff** the CSV: it reads a sample, guesses each column's type and answers instantly, without scanning all 4.4 GB.

</div>

<div style="border-left:6px solid #F43F5E; background:#FFF0F3; color:#881337; padding:14px 18px; border-radius:10px; margin:10px 0; box-shadow:0 1px 3px rgba(0,0,0,.06);">

<b style="font-size:1.02em; letter-spacing:.3px">⚠️ Watch out · The data-type trap</b>

**Some columns *look* numeric but are really identifiers.**
- `Prscrbr_NPI` is a 10-digit doctor ID. Nobody adds IDs together, so we store it as **text**.
- `Prscrbr_State_FIPS` is a state code like `01` (Alabama). Stored as a number, the leading zero disappears and joins with other datasets silently break.

So we **override** those two columns to `VARCHAR` instead of trusting the automatic guess.

</div>

In [2]:
# ── Tell DuckDB which columns are identifiers (text), not numbers ───────────
CSV_TYPES = {"Prscrbr_NPI": "VARCHAR", "Prscrbr_State_FIPS": "VARCHAR"}

# A reusable SQL "table expression" that reads the CSV directly
src = f"read_csv('{RAW_CSV.as_posix()}', header=true, types={CSV_TYPES})"

schema = con.sql(f"DESCRIBE SELECT * FROM {src}").df()[["column_name", "column_type"]]
show(schema, "Schema of the Part D 2024 file (22 columns)")

column_name,column_type
Prscrbr_NPI,VARCHAR
Prscrbr_Last_Org_Name,VARCHAR
Prscrbr_First_Name,VARCHAR
Prscrbr_City,VARCHAR
Prscrbr_State_Abrvtn,VARCHAR
Prscrbr_State_FIPS,VARCHAR
Prscrbr_Type,VARCHAR
Prscrbr_Type_Src,VARCHAR
Brnd_Name,VARCHAR
Gnrc_Name,VARCHAR


In [3]:
# ── Peek at 5 rows ──────────────────────────────────────────────────────────
# Doctor names are public data, but we show doctors aggregated by default, so hide names in previews
preview = con.sql(f"""
    SELECT * EXCLUDE (Prscrbr_Last_Org_Name, Prscrbr_First_Name)
    FROM {src}
    LIMIT 5
""").df()
show(preview, "First 5 rows (names hidden)")

Prscrbr_NPI,Prscrbr_City,Prscrbr_State_Abrvtn,Prscrbr_State_FIPS,Prscrbr_Type,Prscrbr_Type_Src,Brnd_Name,Gnrc_Name,Tot_Clms,Tot_30day_Fills,Tot_Day_Suply,Tot_Drug_Cst,Tot_Benes,GE65_Sprsn_Flag,GE65_Tot_Clms,GE65_Tot_30day_Fills,GE65_Tot_Drug_Cst,GE65_Tot_Day_Suply,GE65_Bene_Sprsn_Flag,GE65_Tot_Benes
1003000126,Bethesda,MD,24,Internal Medicine,Claim-Specialty,Abrysvo,"Rsv Vacc, Pref A And Pref B/Pf",20,20,20,"5,994.0",20,—,20,20,"5,994.0",20,—,20
1003000126,Bethesda,MD,24,Internal Medicine,Claim-Specialty,Boostrix Tdap,"Diphth,pertuss(Acell),tet Vac",12,12,12,648.3,12,—,12,12,648.3,12,—,12
1003000142,Toledo,OH,39,Anesthesiology,Claim-Specialty,Acetaminophen-Codeine,Acetaminophen With Codeine,24,24,628,362.5,11,#,—,—,—,—,*,—
1003000142,Toledo,OH,39,Anesthesiology,Claim-Specialty,Amitriptyline Hcl,Amitriptyline Hcl,47,47,"1,410",271.2,16,—,30,30,150.2,900,*,—
1003000142,Toledo,OH,39,Anesthesiology,Claim-Specialty,Baclofen,Baclofen,47,47,"1,395",333.4,26,—,33,33,222.1,975,#,—


<div style="border-left:6px solid #3B82F6; background:#EAF4FF; color:#1E3A8A; padding:14px 18px; border-radius:10px; margin:10px 0; box-shadow:0 1px 3px rgba(0,0,0,.06);">

<b style="font-size:1.02em; letter-spacing:.3px">📊 Reading the output</b>

- `VARCHAR` = text, `BIGINT` = whole number, `DOUBLE` = decimal number.
- NPI and FIPS now show as `VARCHAR`, so our override worked.
- Blank values (—) in `GE65_*` columns, with a `*` or `#` flag, are **privacy suppression**, explained in section 5.

</div>

<div style="border-left:6px solid #8B5CF6; background:#F3EEFF; color:#3B1F7A; padding:14px 18px; border-radius:10px; margin:10px 0; box-shadow:0 1px 3px rgba(0,0,0,.06);">

## 3 · Count all rows (one full scan)

</div>

<div style="border-left:6px solid #14B8A6; background:#E9FBF8; color:#134E4A; padding:14px 18px; border-radius:10px; margin:10px 0; box-shadow:0 1px 3px rgba(0,0,0,.06);">

<b style="font-size:1.02em; letter-spacing:.3px">🛠️ What the next code does</b>

Reads the whole file once and counts rows. We expect exactly **28,023,892**, the number recorded at download,
so this doubles as a check that the download is complete. `assert` stops the notebook if it isn't.

</div>

In [4]:
t0 = time.time()
n_rows = con.sql(f"SELECT COUNT(*) FROM {src}").fetchone()[0]

print(f"{n_rows:,} rows scanned in {time.time() - t0:.0f} s")
assert n_rows == 28_023_892, "Row count differs from the verified download: the file may be incomplete"

28,023,892 rows scanned in 10 s


<div style="border-left:6px solid #3B82F6; background:#EAF4FF; color:#1E3A8A; padding:14px 18px; border-radius:10px; margin:10px 0; box-shadow:0 1px 3px rgba(0,0,0,.06);">

<b style="font-size:1.02em; letter-spacing:.3px">📊 Reading the output</b>

28 million rows in about **10 seconds** on a laptop. pandas would have needed minutes and probably crashed with an out-of-memory error.

</div>

<div style="border-left:6px solid #8B5CF6; background:#F3EEFF; color:#3B1F7A; padding:14px 18px; border-radius:10px; margin:10px 0; box-shadow:0 1px 3px rgba(0,0,0,.06);">

## 4 · Which drugs count as GLP-1? Discover, don't guess

</div>

<div style="border-left:6px solid #FB923C; background:#FFF4E8; color:#7C2D12; padding:14px 18px; border-radius:10px; margin:10px 0; box-shadow:0 1px 3px rgba(0,0,0,.06);">

<b style="font-size:1.02em; letter-spacing:.3px">💡 Concept</b>

Before filtering, list every generic name that *looks like* a GLP-1 receptor agonist: names ending in **-glutide** or **-enatide**, plus **tirzepatide**.
The filter is then based on **what is actually in the data**, not on memory, and edge cases become visible.

`regexp_matches(text, pattern)` is a *regular expression* search; `|` means "or".

</div>

In [5]:
candidates = con.sql(f"""
    SELECT  Gnrc_Name, Brnd_Name,
            COUNT(*)      AS n_rows,
            SUM(Tot_Clms) AS claims
    FROM    {src}
    WHERE   regexp_matches(lower(Gnrc_Name), 'glutide|enatide|tirzepatide')
    GROUP BY ALL
    ORDER BY claims DESC
""").df()
show(candidates, "Every GLP-1-looking drug in Part D 2024", precision=0)

Gnrc_Name,Brnd_Name,n_rows,claims
Semaglutide,Ozempic,"171,191","9,930,484"
Tirzepatide,Mounjaro,"97,208","4,694,896"
Dulaglutide,Trulicity,"101,479","3,816,376"
Semaglutide,Rybelsus,"35,474","899,704"
Insulin Glargine/Lixisenatide,Soliqua 100-33,"6,568","189,307"
Exenatide Microspheres,Bydureon Bcise,"4,844","85,420"
Semaglutide,Wegovy,"3,956","76,176"
Liraglutide,Victoza 3-Pak,"2,765","44,376"
Insulin Degludec/Liraglutide,Xultophy 100-3.6,"1,110","28,517"
Liraglutide,Victoza 2-Pak,"1,417","27,346"


<div style="border-left:6px solid #3B82F6; background:#EAF4FF; color:#1E3A8A; padding:14px 18px; border-radius:10px; margin:10px 0; box-shadow:0 1px 3px rgba(0,0,0,.06);">

<b style="font-size:1.02em; letter-spacing:.3px">📊 Reading the output</b>

- **Ozempic** leads (9.9M claims), then **Mounjaro** (4.7M) and **Trulicity** (3.8M).
- **Zepbound** has only **14 claims**: Medicare didn't pay for obesity drugs in 2024 (more in section 6).
- Two surprises are worth a closer look: `Exenatide Microspheres` and `Teduglutide`.

</div>

<div style="border-left:6px solid #F43F5E; background:#FFF0F3; color:#881337; padding:14px 18px; border-radius:10px; margin:10px 0; box-shadow:0 1px 3px rgba(0,0,0,.06);">

<b style="font-size:1.02em; letter-spacing:.3px">⚠️ Watch out · Two traps the discovery step caught</b>

**Trap 1, a different name:** Bydureon BCise is listed as `Exenatide Microspheres`, not `Exenatide`.
An exact match on "exenatide" alone silently drops **4,844 rows**. Always look at the real values before you filter.

**Trap 2, a false positive:** `Teduglutide` (Gattex) matches the pattern but is a **GLP-2** drug for short bowel syndrome, not a GLP-1. → excluded.

**A judgement call:** `Insulin Glargine/Lixisenatide` (Soliqua) is a basal-insulin-first combination, small and outside the tirzepatide-vs-semaglutide story. → excluded.
Write such choices down: *"why did you include X but not Y?"* is a question every data choice should be able to answer.

</div>

In [6]:
# ── Final filter list (lower-case, matched with lower(trim(...)) below) ─────
GLP1_GENERICS = [
    "tirzepatide",                    # Mounjaro, Zepbound            (Lilly)
    "semaglutide",                    # Ozempic, Wegovy, Rybelsus     (Novo Nordisk)
    "dulaglutide",                    # Trulicity                     (Lilly)
    "liraglutide",                    # Victoza, generics             (Novo Nordisk + generics)
    "exenatide",                      # Byetta                        (AstraZeneca)
    "exenatide microspheres",         # Bydureon BCise: different generic name!
    "insulin degludec/liraglutide",   # Xultophy                      (Novo Nordisk)
]
generic_list_sql = ", ".join(f"'{g}'" for g in GLP1_GENERICS)
print(f"{len(GLP1_GENERICS)} generic names in the filter")

7 generic names in the filter


<div style="border-left:6px solid #8B5CF6; background:#F3EEFF; color:#3B1F7A; padding:14px 18px; border-radius:10px; margin:10px 0; box-shadow:0 1px 3px rgba(0,0,0,.06);">

## 5 · Filter → Parquet, then validate

</div>

<div style="border-left:6px solid #FB923C; background:#FFF4E8; color:#7C2D12; padding:14px 18px; border-radius:10px; margin:10px 0; box-shadow:0 1px 3px rgba(0,0,0,.06);">

<b style="font-size:1.02em; letter-spacing:.3px">💡 Concept</b>

`COPY (query) TO 'file.parquet'` streams matching rows **straight to disk**, so pandas never sees 28M rows.

**Why Parquet instead of CSV?**
- **Columnar**: reading 2 of 22 columns touches only those 2 on disk
- **Compressed** (ZSTD): our file is **~400× smaller** than the CSV
- **Typed**: text vs number is stored *inside* the file, so no re-guessing and no lost leading zeros
- Read natively by pandas, DuckDB, Spark, Polars and Hugging Face `datasets`

**Why `lower(trim(...))`?** Public files often have stray spaces or mixed case; normalising both sides makes the match robust.

</div>

In [7]:
OUT_PARQUET.parent.mkdir(parents=True, exist_ok=True)
t0 = time.time()

con.sql(f"""
    COPY (
        SELECT  *,
                trim(Brnd_Name) AS Brand,          -- cleaned copies used in all analysis below
                trim(Gnrc_Name) AS Generic
        FROM    {src}
        WHERE   lower(trim(Gnrc_Name)) IN ({generic_list_sql})
        ORDER BY Prscrbr_NPI, Brand                -- same doctor's rows together: better compression + faster joins
    ) TO '{OUT_PARQUET.as_posix()}' (FORMAT parquet, COMPRESSION zstd)
""")

size_mb = OUT_PARQUET.stat().st_size / 1e6
print(f"Wrote {OUT_PARQUET.relative_to(ROOT)}: {size_mb:.1f} MB in {time.time() - t0:.0f} s "
      f"(the CSV is {RAW_CSV.stat().st_size / 1e9:.2f} GB)")

Wrote data\02_interim\part_d_2024_glp1.parquet: 11.4 MB in 17 s (the CSV is 4.43 GB)


<div style="border-left:6px solid #14B8A6; background:#E9FBF8; color:#134E4A; padding:14px 18px; border-radius:10px; margin:10px 0; box-shadow:0 1px 3px rgba(0,0,0,.06);">

<b style="font-size:1.02em; letter-spacing:.3px">🛠️ What the next code does</b>

From here on we read only the small Parquet file, and every query takes milliseconds.
Check the totals against the numbers recorded on 18 Oct 2026: **420,364 rows** and **199,831 prescribers**.

</div>

In [8]:
glp1 = f"read_parquet('{OUT_PARQUET.as_posix()}')"     # table expression for the subset

check = con.sql(f"""
    SELECT  COUNT(*)                     AS rows,
            COUNT(DISTINCT Prscrbr_NPI)  AS prescribers,
            COUNT(DISTINCT Brand)        AS brands,
            SUM(Tot_Clms)                AS claims,
            SUM(Tot_Drug_Cst) / 1e9      AS cost_usd_bn
    FROM    {glp1}
""").df()

assert (check.loc[0, "rows"], check.loc[0, "prescribers"]) == (420_364, 199_831), "Subset differs from verified numbers"
show(check, "GLP-1 subset: headline numbers")

rows,prescribers,brands,claims,cost_usd_bn
"420,364","199,831",12,"19,620,006",24.6


<div style="border-left:6px solid #3B82F6; background:#EAF4FF; color:#1E3A8A; padding:14px 18px; border-radius:10px; margin:10px 0; box-shadow:0 1px 3px rgba(0,0,0,.06);">

<b style="font-size:1.02em; letter-spacing:.3px">📊 Reading the output</b>

**420,364 rows · 199,831 prescribers · 19.6M claims · US$24.6 bn** of Medicare spending on GLP-1s in 2024. Both checks pass ✅.

</div>

<div style="border-left:6px solid #FB923C; background:#FFF4E8; color:#7C2D12; padding:14px 18px; border-radius:10px; margin:10px 0; box-shadow:0 1px 3px rgba(0,0,0,.06);">

<b style="font-size:1.02em; letter-spacing:.3px">💡 Concept · Three data-quality checks</b>

1. **Uniqueness:** exactly one row per (prescriber, brand, generic). Duplicates would double-count after joins.
2. **Sanity:** no negative claims or costs.
3. **Missing values:** CMS **suppresses** small numbers to protect patient privacy:
   - `Tot_Benes` is blank when **fewer than 11 patients**
   - `GE65_*` flags: `*` = suppressed (<11); `#` = suppressed so a small number can't be worked out by subtraction

   Blanks here are **by design, not errors**, and must **never be filled with 0**.

</div>

In [9]:
# ── 1 + 2: uniqueness and sanity ────────────────────────────────────────────
dupes = con.sql(f"""
    SELECT COUNT(*) FROM (
        SELECT Prscrbr_NPI, Brand, Generic
        FROM {glp1}
        GROUP BY ALL
        HAVING COUNT(*) > 1
    )
""").fetchone()[0]
negatives = con.sql(f"SELECT COUNT(*) FROM {glp1} WHERE Tot_Clms < 0 OR Tot_Drug_Cst < 0").fetchone()[0]

print(f"Duplicate (NPI, brand, generic) keys : {dupes}")
print(f"Rows with negative claims or cost   : {negatives}")

# ── 3: how much is suppressed? ──────────────────────────────────────────────
missing = con.sql(f"""
    SELECT  ROUND(100 * AVG((Tot_Benes      IS NULL)::INT), 1) AS pct_patients_suppressed,
            ROUND(100 * AVG((GE65_Tot_Benes IS NULL)::INT), 1) AS pct_65plus_patients_missing,
            ROUND(100 * AVG((Prscrbr_Type   IS NULL)::INT), 1) AS pct_specialty_missing
    FROM    {glp1}
""").df()
show(missing, "Share of rows with blank values (%)")

Duplicate (NPI, brand, generic) keys : 0
Rows with negative claims or cost   : 0


pct_patients_suppressed,pct_65plus_patients_missing,pct_specialty_missing
71.3,94.9,0


<div style="border-left:6px solid #3B82F6; background:#EAF4FF; color:#1E3A8A; padding:14px 18px; border-radius:10px; margin:10px 0; box-shadow:0 1px 3px rgba(0,0,0,.06);">

<b style="font-size:1.02em; letter-spacing:.3px">📊 Reading the output</b>

- **0 duplicates, 0 negatives**: the table is clean.
- **71%** of rows have a suppressed patient count: most doctors treat fewer than 11 Medicare patients per drug. So we'll use **claims** (always present) as the main volume measure, not patients.
- Specialty is never missing, which is good for segmentation.

</div>

<div style="border-left:6px solid #8B5CF6; background:#F3EEFF; color:#3B1F7A; padding:14px 18px; border-radius:10px; margin:10px 0; box-shadow:0 1px 3px rgba(0,0,0,.06);">

## 6 · First profile of the 2024 Medicare GLP-1 market

</div>

<div style="border-left:6px solid #F43F5E; background:#FFF0F3; color:#881337; padding:14px 18px; border-radius:10px; margin:10px 0; box-shadow:0 1px 3px rgba(0,0,0,.06);">

<b style="font-size:1.02em; letter-spacing:.3px">⚠️ Watch out</b>

CMS **drops rows with fewer than 11 claims** entirely, so `prescribers` below means doctors with **≥ 11 claims** of that drug.
Low-volume prescribers are invisible: a limitation to state in the README.

</div>

In [10]:
by_brand = con.sql(f"""
    SELECT  Brand, Generic,
            SUM(Tot_Clms)                               AS claims,
            SUM(Tot_Drug_Cst) / 1e6                     AS cost_usd_m,
            COUNT(DISTINCT Prscrbr_NPI)                 AS prescribers,
            SUM(Tot_Drug_Cst) / SUM(Tot_30day_Fills)    AS cost_per_30day_fill
    FROM    {glp1}
    GROUP BY ALL
    ORDER BY claims DESC
""").df()
by_brand["claims_share_pct"] = 100 * by_brand["claims"] / by_brand["claims"].sum()
show(by_brand, "6a · Market by brand")

Brand,Generic,claims,cost_usd_m,prescribers,cost_per_30day_fill,claims_share_pct
Ozempic,Semaglutide,"9,930,484","12,376.0","171,191",904.6,50.6
Mounjaro,Tirzepatide,"4,694,896","5,825.3","97,208","1,039.5",23.9
Trulicity,Dulaglutide,"3,816,376","4,786.8","101,479",946.9,19.5
Rybelsus,Semaglutide,"899,704","1,309.3","35,474",916.2,4.6
Bydureon Bcise,Exenatide Microspheres,"85,420",89.3,"4,844",797.1,0.4
Wegovy,Semaglutide,"76,176",107.3,"3,956","1,328.5",0.4
Victoza 3-Pak,Liraglutide,"44,376",44.7,"2,765",722.3,0.2
Xultophy 100-3.6,Insulin Degludec/Liraglutide,"28,517",42.9,"1,110",937.4,0.1
Victoza 2-Pak,Liraglutide,"27,346",17.6,"1,417",583.8,0.1
Liraglutide,Liraglutide,"14,832",9.7,771,586.2,0.1


<div style="border-left:6px solid #3B82F6; background:#EAF4FF; color:#1E3A8A; padding:14px 18px; border-radius:10px; margin:10px 0; box-shadow:0 1px 3px rgba(0,0,0,.06);">

<b style="font-size:1.02em; letter-spacing:.3px">📊 Reading the output</b>

- **Ozempic ~51%**, **Mounjaro ~24%**, **Trulicity ~19%** of claims: three brands make up ~94% of the market.
- A 30-day fill costs **~US$900–1,300** before rebates.
- **Zepbound: 14 claims from 1 prescriber.** It's the same molecule as Mounjaro (tirzepatide), but approved for **obesity**, which Medicare didn't cover in 2024.

</div>

<div style="border-left:6px solid #10B981; background:#EAFBF1; color:#065F46; padding:14px 18px; border-radius:10px; margin:10px 0; box-shadow:0 1px 3px rgba(0,0,0,.06);">

<b style="font-size:1.02em; letter-spacing:.3px">🎯 Story hook</b>

The **Medicare GLP-1 Bridge (July 2026)** opens obesity coverage, so the doctors who already prescribe Mounjaro and Ozempic to older patients are the natural first list for obesity targeting.
Notebook 05 turns this into a score.

</div>

In [11]:
by_mfr = con.sql(f"""
    SELECT  CASE
                WHEN Generic IN ('Tirzepatide', 'Dulaglutide')                                 THEN 'Eli Lilly'
                WHEN Generic IN ('Semaglutide', 'Liraglutide', 'Insulin Degludec/Liraglutide') THEN 'Novo Nordisk*'
                ELSE 'Other (exenatide: AstraZeneca)'
            END                                                          AS manufacturer,
            SUM(Tot_Clms)                                                AS claims,
            ROUND(100 * SUM(Tot_Clms) / SUM(SUM(Tot_Clms)) OVER (), 1)   AS claims_share_pct,
            SUM(Tot_Drug_Cst) / 1e6                                      AS cost_usd_m
    FROM    {glp1}
    GROUP BY ALL
    ORDER BY claims DESC
""").df()
show(by_mfr, "6b · Market by manufacturer (molecule level)")

manufacturer,claims,claims_share_pct,cost_usd_m
Novo Nordisk*,"11,021,435",56.2,"13,907.5"
Eli Lilly,"8,511,286",43.4,"10,612.1"
Other (exenatide: AstraZeneca),"87,285",0.4,90.9


<div style="border-left:6px solid #3B82F6; background:#EAF4FF; color:#1E3A8A; padding:14px 18px; border-radius:10px; margin:10px 0; box-shadow:0 1px 3px rgba(0,0,0,.06);">

<b style="font-size:1.02em; letter-spacing:.3px">📊 Reading the output</b>

**Novo Nordisk ~56% vs Eli Lilly ~43%** of claims. (\* Generic liraglutide from other makers is grouped with Novo for simplicity; it's tiny.)

`SUM(SUM(x)) OVER ()` is a **window function**: it computes the grand total next to each group, so we can get shares in one query.

</div>

In [12]:
by_specialty = con.sql(f"""
    SELECT  Prscrbr_Type                                                 AS specialty,
            COUNT(DISTINCT Prscrbr_NPI)                                  AS prescribers,
            SUM(Tot_Clms)                                                AS claims,
            ROUND(100 * SUM(Tot_Clms) / SUM(SUM(Tot_Clms)) OVER (), 1)   AS claims_share_pct,
            ROUND(SUM(Tot_Clms) / COUNT(DISTINCT Prscrbr_NPI), 0)        AS claims_per_prescriber
    FROM    {glp1}
    GROUP BY ALL
    ORDER BY claims DESC
    LIMIT 15
""").df()
show(by_specialty, "6c · Top 15 specialties", precision=0)

specialty,prescribers,claims,claims_share_pct,claims_per_prescriber
Family Practice,"64,038","5,794,176",30,90
Internal Medicine,"44,984","4,890,255",25,109
Nurse Practitioner,"51,791","4,050,021",21,78
Endocrinology,"5,881","2,504,082",13,426
Physician Assistant,"17,518","1,376,160",7,79
General Practice,"2,385","190,263",1,80
Cardiology,"2,309","165,279",1,72
Pharmacist,"1,022","116,413",1,114
Nephrology,"1,148","83,642",0,73
Geriatric Medicine,795,"54,687",0,69


<div style="border-left:6px solid #3B82F6; background:#EAF4FF; color:#1E3A8A; padding:14px 18px; border-radius:10px; margin:10px 0; box-shadow:0 1px 3px rgba(0,0,0,.06);">

<b style="font-size:1.02em; letter-spacing:.3px">📊 Reading the output</b>

- **Primary care dominates volume:** Family Practice + Internal Medicine + Nurse Practitioners ≈ **75%** of claims.
- **Endocrinologists are few (5,879) but intense:** **424 claims each**, about **5×** a family doctor (~90).
- Sales-optimisation insight: endocrinologists are high-value individual targets, while primary care is where the **breadth** is. These are two different coverage strategies.

</div>

In [13]:
tirz_vs_sema = con.sql(f"""
    WITH s AS (
        SELECT  Prscrbr_Type                                            AS specialty,
                SUM(Tot_Clms) FILTER (WHERE Generic = 'Tirzepatide')    AS tirzepatide,
                SUM(Tot_Clms) FILTER (WHERE Generic = 'Semaglutide')    AS semaglutide,
                SUM(Tot_Clms)                                           AS all_glp1
        FROM    {glp1}
        GROUP BY ALL
    )
    SELECT  *,
            ROUND(100 * tirzepatide / (tirzepatide + semaglutide), 1)   AS tirz_share_pct
    FROM    s
    WHERE   all_glp1 > 100000
    ORDER BY all_glp1 DESC
""").df()
show(tirz_vs_sema, "6d · Tirzepatide share vs semaglutide, by specialty", precision=0)

specialty,tirzepatide,semaglutide,all_glp1,tirz_share_pct
Family Practice,"1,299,946","3,293,689","5,794,176",28
Internal Medicine,"1,117,261","2,779,475","4,890,255",29
Nurse Practitioner,"1,038,260","2,191,633","4,050,021",32
Endocrinology,"694,945","1,293,016","2,504,082",35
Physician Assistant,"333,034","759,480","1,376,160",30
General Practice,"36,509","102,778","190,263",26
Cardiology,"42,684","110,507","165,279",28
Pharmacist,"16,266","71,307","116,413",19


<div style="border-left:6px solid #3B82F6; background:#EAF4FF; color:#1E3A8A; padding:14px 18px; border-radius:10px; margin:10px 0; box-shadow:0 1px 3px rgba(0,0,0,.06);">

<b style="font-size:1.02em; letter-spacing:.3px">📊 Reading the output</b>

`tirz_share_pct` = tirzepatide ÷ (tirzepatide + semaglutide).
Tirzepatide does best with **endocrinologists (35%)** and **nurse practitioners (32%)**, and worst with **pharmacists (19%)** and **general practice (26%)**: specialists adopt new molecules first.

</div>

In [14]:
by_state = con.sql(f"""
    SELECT  Prscrbr_State_Abrvtn                                                  AS state,
            COUNT(DISTINCT Prscrbr_NPI)                                           AS prescribers,
            SUM(Tot_Clms)                                                         AS claims,
            ROUND(100 * SUM(Tot_Clms) FILTER (WHERE Generic = 'Tirzepatide')
                      / SUM(Tot_Clms) FILTER (WHERE Generic IN ('Tirzepatide', 'Semaglutide')), 1)
                                                                                  AS tirz_share_pct
    FROM    {glp1}
    GROUP BY ALL
    ORDER BY claims DESC
    LIMIT 15
""").df()
show(by_state, "6e · Top 15 states by claims")

state,prescribers,claims,tirz_share_pct
TX,"14,689","1,659,229",32.5
CA,"19,330","1,647,966",18.9
NY,"12,442","1,327,074",26.5
FL,"13,230","1,306,286",32.8
PA,"8,738","925,711",25.9
OH,"7,743","766,117",30.1
NC,"7,339","756,282",27.9
GA,"5,625","696,815",35.0
MI,"7,105","689,268",38.7
IL,"6,802","622,137",26.3


<div style="border-left:6px solid #3B82F6; background:#EAF4FF; color:#1E3A8A; padding:14px 18px; border-radius:10px; margin:10px 0; box-shadow:0 1px 3px rgba(0,0,0,.06);">

<b style="font-size:1.02em; letter-spacing:.3px">📊 Reading the output</b>

Big regional differences: tirzepatide share is **44% in Louisiana** and **41% in Alabama**, but only **19% in California**.
Same drug, same year, so something local (payer formularies, sales coverage, specialist mix) drives it. That "why" is exactly what the AI assistant should help explain later.

</div>

<div style="border-left:6px solid #FB923C; background:#FFF4E8; color:#7C2D12; padding:14px 18px; border-radius:10px; margin:10px 0; box-shadow:0 1px 3px rgba(0,0,0,.06);">

<b style="font-size:1.02em; letter-spacing:.3px">💡 Concept · Deciling</b>

Sales teams **decile** doctors: rank them by volume and cut them into 10 equal-sized groups (decile 1 = top 10%).
If the top deciles write most of the claims, a small sales force can cover most of the market: the **Pareto (80/20)** idea, and the first step of targeting.
`NTILE(10) OVER (ORDER BY claims DESC)` does the cutting in SQL.

</div>

In [15]:
deciles = con.sql(f"""
    WITH doc AS (                                   -- one row per doctor
        SELECT Prscrbr_NPI, SUM(Tot_Clms) AS claims
        FROM   {glp1}
        GROUP BY 1
    ),
    ranked AS (                                     -- assign deciles 1..10
        SELECT *, NTILE(10) OVER (ORDER BY claims DESC) AS decile
        FROM   doc
    )
    SELECT  decile,
            COUNT(*)                                                                     AS prescribers,
            SUM(claims)                                                                  AS claims,
            ROUND(100 * SUM(claims) / SUM(SUM(claims)) OVER (), 1)                       AS claims_share_pct,
            ROUND(100 * SUM(SUM(claims)) OVER (ORDER BY decile) / SUM(SUM(claims)) OVER (), 1) AS cumulative_pct
    FROM    ranked
    GROUP BY decile
    ORDER BY decile
""").df()
show(deciles, "6f · Prescriber deciles (1 = highest volume)")

decile,prescribers,claims,claims_share_pct,cumulative_pct
1,"19,984","8,125,461",41.4,41.4
2,"19,983","3,415,381",17.4,58.8
3,"19,983","2,340,193",11.9,70.7
4,"19,983","1,722,503",8.8,79.5
5,"19,983","1,289,020",6.6,86.1
6,"19,983","958,230",4.9,91.0
7,"19,983","701,372",3.6,94.6
8,"19,983","495,371",2.5,97.1
9,"19,983","331,423",1.7,98.8
10,"19,983","241,052",1.2,100.0


<div style="border-left:6px solid #3B82F6; background:#EAF4FF; color:#1E3A8A; padding:14px 18px; border-radius:10px; margin:10px 0; box-shadow:0 1px 3px rgba(0,0,0,.06);">

<b style="font-size:1.02em; letter-spacing:.3px">📊 Reading the output</b>

- **Top 10% of prescribers (~20k doctors) write 41% of all GLP-1 claims.**
- **Top 40% write ~80%**: a textbook Pareto curve.
- The bottom half writes only ~14%, so visiting them all would be expensive for little volume.

</div>

<div style="border-left:6px solid #8B5CF6; background:#F3EEFF; color:#3B1F7A; padding:14px 18px; border-radius:10px; margin:10px 0; box-shadow:0 1px 3px rgba(0,0,0,.06);">

## 7 · Summary & next steps

</div>

<div style="border-left:6px solid #F43F5E; background:#FFF0F3; color:#881337; padding:14px 18px; border-radius:10px; margin:10px 0; box-shadow:0 1px 3px rgba(0,0,0,.06);">

<b style="font-size:1.02em; letter-spacing:.3px">⚠️ Limitations to carry forward</b>

1. **Medicare Part D only** (mostly 65+): no commercial insurance, cash-pay or Medicaid at doctor level.
2. Rows with **< 11 claims are dropped** by CMS, so low-volume prescribers are invisible.
3. `Tot_Drug_Cst` is the **gross** amount before rebates, not company revenue.
4. Obesity use (Zepbound, Wegovy) is almost absent in 2024, so the 2026 opportunity must be **modelled**, not observed.

</div>

<div style="border-left:6px solid #10B981; background:#EAFBF1; color:#065F46; padding:14px 18px; border-radius:10px; margin:10px 0; box-shadow:0 1px 3px rgba(0,0,0,.06);">

<b style="font-size:1.02em; letter-spacing:.3px">🎯 Key takeaways</b>

**Done:** 28M rows → validated GLP-1 subset (`data/02_interim/part_d_2024_glp1.parquet`), plus a first market profile.
**Next:** Notebook 01 collects the other public sources, and notebook 03 joins everything on NPI into the one-row-per-doctor **HCP 360** table.

**Check your understanding**
- *Why DuckDB over pandas for a 4 GB file?* Streaming, columnar, vectorised, parallel; reduce first, then use pandas.
- *Why Parquet?* Columnar, compressed, typed, works across tools.
- *Why store NPI and FIPS as text?* They're identifiers, and leading zeros matter.
- *What does a blank `Tot_Benes` mean?* Privacy suppression (< 11 patients), not zero; never impute 0.
- *How did you choose which drugs to include?* Discovered candidates with a regex, then documented every include/exclude.
- *What does deciling tell a sales team?* How concentrated prescribing is (top 10% → 41%), which sets coverage strategy.

</div>